In [66]:
import pandas as pd
icpe = pd.read_csv('Nomenclature_ICPE.csv', sep=';', dtype=str)
icpe.head()

,Code,Description,Service
0,1xxx,SUBSTANCES,NaN
1,11xx,Gaz à effet de serre,NaN
2,1185,Gaz à effet de serre fluorés,SSEEC-BPC
3,13xx,Explosifs et substances explosibles,NaN
4,131x,Explosifs,NaN


In [67]:
icpe = icpe[icpe['Service'].notna()]
icpe = icpe.drop(columns=['Service'])
icpe.head()

,Code,Description
2,1185,Gaz à effet de serre fluorés
5,1312,Mise en œuvre de produits explosifs à des fins...
8,1413,Installations de remplissage de réservoirs de ...
9,1414,Installations de remplissage ou de distributio...
10,1416,Station services (hydrogène)


In [68]:
import charset_normalizer
with open('nom_composant.csv', 'rb') as f:
    print(charset_normalizer.detect(f.read(10000)))

{'encoding': 'mac_iceland', 'language': 'French', 'confidence': 1.0}


In [69]:
composants = pd.read_csv('nom_composant.csv', sep=';', dtype=str, encoding='mac_iceland')
composants.head(10)


,nomComposant
0,0-15 degré Celsius
1,100-149 degré Celsius
2,15-40 degré Celsius
3,150-249 degré Celsius
4,250-499 degré Celsius
5,40-99 degré Celsius
6,< 0 degré Celsius
7,> 500 degré Celsius
8,Acide acétique
9,Acide arsenic


## Prompt ClaudePRO ##

Voici deux documents : nom_composants.csv est une liste de composant ou de conditions expérimentales, ICPE.csv est la liste de toutes les rubriques ICPE ainsi que leur libelle. Je veux que tu me retournes une table csv qui liste les composants appartenants aux catégories ICPE. 

Si le lien est explicite établit la correspondance entre le numéro ICPE et les composants. Si le lien n'est pas clair ou que le libele est trop flou n'établit pas la correspondance. Dans ce cas créé deux nouvelles tables : une dans laquelle tu suggères la création de nouveaux composants et une autre dans laquelle tu listes les numéros ICPE pour lesquels tu n'as établi aucune correspondance. 

REPONSE : 

Bien, j'ai maintenant une bonne compréhension des deux fichiers. Il y a 405 composants et 265 rubriques ICPE. Laissez-moi construire les correspondances de manière systématique.

Voici les 3 tables. Sur 265 rubriques ICPE : 69 avec correspondance, 35 substances à créer, 161 sans correspondance.

**• Table 1 — 192 correspondances explicites (69 rubriques × 121 composants), organisée en 3 blocs :**

- Substances Seveso nommées (4700+) : correspondance directe substance ↔ code, ex. 4735 ↔ Ammoniac, 4715 ↔ Hydrogène, 4710 ↔ Chlore, 4711 ↔ sels de nickel (Chlorure/Oxyde/Sulfate/Fluoborate de nickel), 4734 ↔ carburants (Essence, Gazole, Fioul, Pétrole brut, e-diesel…), 4801 ↔ Charbon/Coke.
- Rubriques matériau (1000/2000/3000) dont le libellé nomme explicitement la matière : 1530 papiers/cartons, 1531-1532 bois, 1630 soude/potasse, 2240 huiles et corps gras, 2520/3310 ciment-chaux-plâtre, 2545/3220 acier-fonte, 2530/3330 verre, 2662 polymères, 2714 déchets papier/plastique/bois/textile, etc.
- Composant générique explicitement rattaché : 1978 ↔ Solvant, 2718/2790/3510 ↔ Déchets dangereux, 2971 ↔ Combustible solide de récupération, 2960/2970/3690 ↔ Dioxyde de carbone, 3710 ↔ Eau usée.


**• Table 2 — 35 composants à créer, principalement des substances Seveso spécifiques absentes de la liste :**
Brome, Fluor, Arsine, Phosphine, phosgène, Oxyde d'éthylène, Oxyde de propylène, Nitrate d'ammonium (3 variantes), Perchlorate d'ammonium, Trifluorure de bore, dioxines/PCDF, Carbure de calcium, PCB, acrylates de méthyle/tert-butyle, plusieurs amines et pyridines. À noter : je n'ai pas rattaché 4707/4708 (Pentoxyde/Trioxyde d'arsenic) à « Oxyde d'arsenic » (générique) — mieux vaut créer les deux formes chimiques distinctes.

**• Table 3 — 161 rubriques sans correspondance, classées par motif :**

- Rubriques d'activité pure sans matière rattachable (élevages 2101-2150, ateliers textile/cuir 2311-2360, tanneries, ateliers métaux 2560-2567, chimie 3410-3450, imprimeries, blanchisseries…).
- Classes de danger CLP (4110-4150 toxicité, 4310-4331 inflammabilité, 4410-4442 auto-réactifs/comburants, 4510-4630 danger environnemental) — ces codes reposent sur la classification CLP de chaque substance, information non fournie ; une correspondance nécessiterait de croiser avec les fiches de données de sécurité de chaque composant.
- Rubriques radioactives (1716, 1735, 2797, 2798) — aucun composant radioactif dans la liste.
- Rubriques explosifs (1312, 3460, 4210, 4220, 4240, 2793) — aucun explosif listé.

**Points d'attention méthodologiques :**

- Les rubriques CLP (4110-4630) sont volontairement laissées en Table 3 : y rattacher automatiquement toutes les substances toxiques ou inflammables serait spéculatif. Si vous disposez des classifications CLP de chaque composant, un second passage permettrait de compléter.
- « Isocyanate » (générique) n'est pas rattaché à 4724 (Isocyanate de méthyle) ni à 4726 (2,4-diisocyanate de toluène) car trop imprécis — j'ai plutôt suggéré la création des deux substances spécifiques.
- « Alcool » (générique) n'est pas rattaché à 4755 (Alcools de bouche d'origine agricole) pour la même raison.
- « Nickel » (métal) n'est pas rattaché à 4711 (Composés de nickel) — seuls les sels le sont, conformément à l'esprit de la rubrique Seveso.



In [70]:
correspondance = pd.read_csv('reponse_claude/table1_correspondances.csv')
a_creer = pd.read_csv('reponse_claude/table2_composants_a_creer.csv')
sans_corr = pd.read_csv('reponse_claude/table3_icpe_sans_correspondance.csv')

print(correspondance.shape, a_creer.shape, sans_corr.shape)

(192, 3) (35, 3) (161, 3)


In [75]:
import pandas as pd

pd.set_option('display.max_rows', None)       # Affiche TOUTES les lignes
pd.set_option('display.max_columns', None)    # Affiche TOUTES les colonnes
pd.set_option('display.width', None)          # Désactive la limite de largeur de ligne
pd.set_option('display.max_colwidth', None)   # Affiche le texte des colonnes en entier (sans ...)


In [72]:
correspondance.head(192)

,Code_ICPE,Libelle_ICPE,Composant
0,1530,"Dépôts de papiers, cartons ou matériaux combustibles analogues",Papier
1,1530,"Dépôts de papiers, cartons ou matériaux combustibles analogues",Carton
2,1531,"Stockages, par voie humide (immersion ou aspersion), de bois non traité chimiquement",Bois en fin de vie propre (palette)
3,1531,"Stockages, par voie humide (immersion ou aspersion), de bois non traité chimiquement",Bois et sous-produits du bois
4,1532,Stockage de bois ou de matériaux combustibles analogues,Bois en fin de vie propre (palette)
5,1532,Stockage de bois ou de matériaux combustibles analogues,Bois et sous-produits du bois
6,1532,Stockage de bois ou de matériaux combustibles analogues,Bois traité et/ou dangereux
7,1630,Emploi ou stockage de lessives de soude ou de potasse caustique,Soude
8,1630,Emploi ou stockage de lessives de soude ou de potasse caustique,Potasse
9,1630,Emploi ou stockage de lessives de soude ou de potasse caustique,Hydroxyde de sodium


In [73]:
a_creer.head(35)

,Code_ICPE,Libelle_ICPE,Composant_suggere
0,1185,Gaz à effet de serre fluorés,"Gaz à effet de serre fluorés (ex : HFC, PFC, SF6)"
1,1455,Stockage de carbure de calcium,Carbure de calcium
2,2792,Traitement de déchets contenant des PCB,Déchet contenant des PCB (polychlorobiphényles)
3,4707,Pentoxyde d'arsenic,Pentoxyde d'arsenic
4,4708,Trioxyde d'arsenic,Trioxyde d'arsenic
5,4709,Brome,Brome
6,4712,Ethylèneimine,Ethylèneimine
7,4713,Fluor,Fluor
8,4717,Plombs alkyls,Plombs alkyls
9,4720,Oxyde d'éthylène,Oxyde d'éthylène


In [74]:
sans_corr.head(161)

,Code_ICPE,Libelle_ICPE,Raison
0,1312,Mise en œuvre de produits explosifs à des fins industrielles,Classe/activité explosifs — aucun explosif dans la liste des composants
1,1413,"Installations de remplissage de réservoirs de gaz naturel ou biogaz, sous pression",Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
2,1414,Installations de remplissage ou de distribution de gaz inflammables liquéfiés,Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
3,1416,Station services (hydrogène),Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
4,1421,Installation de remplissage d'aérosols inflammables,Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
5,1434,Installations de remplissage ou de distribution de liquides inflammables,Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
6,1435,Stations-service,Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
7,1436,Liquides de point éclair compris entre 60°C et 93°C,Catégorie définie par le point éclair — dépend de la propriété physique de chaque liquide
8,1450,Solides inflammables,Catégorie 'solides inflammables' — dépend de la classification CLP
9,1510,"Stockage de matières, produits ou substances combustibles dans des entrepôts couverts",Rubrique d'activité — libellé sans substance spécifique rattachable à la liste
